# 🎓 AI Learn — Random Forest Model #2 (Platform-Native)
### Trained on application-generated features — NOT xAPI
This model uses the exact feature set your FastAPI backend already produces:
`quiz_accuracy, avg_time_per_question, total_attempts, videos_watched, chatbot_questions,
study_duration, daily_streak, ewma_accuracy, prerequisite_mastery, previous_attempt_accuracy`

Target: `label` → Weak / Moderate / Strong

---

## Step 1 — Install & Import Libraries

In [ ]:
!pip install pandas numpy scikit-learn joblib matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix
)
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
print('✅ Libraries imported')

## Step 2 — Upload Dataset
Upload `ai_learn_performance_dataset.csv` (the synthetic dataset generated to match your platform's schema).

In [ ]:
from google.colab import files
uploaded = files.upload()   # select ai_learn_performance_dataset.csv
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'Shape: {df.shape}')
df.head()

## Step 3 — Explore Dataset

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Label Distribution ===')
print(df['label'].value_counts())
print()
print((df['label'].value_counts(normalize=True) * 100).round(1))

fig, ax = plt.subplots(figsize=(6, 4))
df['label'].value_counts().reindex(['Weak','Moderate','Strong']).plot(
    kind='bar', ax=ax, color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='black'
)
ax.set_title('Label Distribution (Weak / Moderate / Strong)', fontsize=14, fontweight='bold')
ax.set_xlabel('Label'); ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('outputs/label_distribution.png', dpi=150)
plt.show()
print('✅ Label distribution chart saved')

## Step 4 — Clean Dataset

In [ ]:
before = len(df)
df = df.drop_duplicates()
df = df.dropna()
print(f'Rows before: {before}  |  after: {len(df)}  |  removed: {before - len(df)}')

## Step 5 — Feature Selection
We deliberately EXCLUDE `user_id` and `topic` (text/identifier — not predictive) and `subject` (redundant with `topic_id`).
`question_difficulty` is label-encoded since it's categorical (Easy/Medium/Hard).

In [ ]:
difficulty_encoder = LabelEncoder()
df['question_difficulty_enc'] = difficulty_encoder.fit_transform(df['question_difficulty'])
print('Difficulty mapping:', dict(zip(difficulty_encoder.classes_, difficulty_encoder.transform(difficulty_encoder.classes_))))

FEATURE_COLUMNS = [
    'topic_id',
    'quiz_accuracy',
    'avg_time_per_question',
    'total_attempts',
    'question_difficulty_enc',
    'videos_watched',
    'articles_read',
    'chatbot_questions',
    'study_duration',
    'daily_streak',
    'ewma_accuracy',
    'prerequisite_mastery',
    'previous_attempt_accuracy',
]

X = df[FEATURE_COLUMNS]
y = df['label']

print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Feature columns: {FEATURE_COLUMNS}')

## Step 6 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

## Step 7 — Train Random Forest
`class_weight='balanced'` is used since label distribution is naturally imbalanced (more Weak than Strong students), matching real platform behavior.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)
print('✅ Model trained')

## Step 8 — Evaluate Model

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f'\n🎯 Accuracy: {accuracy * 100:.2f}%')
print('\n--- Classification Report ---')
report = classification_report(y_test, predictions, labels=['Weak','Moderate','Strong'])
print(report)

with open('outputs/classification_report.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy * 100:.2f}%\n\n')
    f.write(report)
print('✅ Saved outputs/classification_report.txt')

## Step 9 — Confusion Matrix

In [ ]:
labels = ['Weak', 'Moderate', 'Strong']
cm = confusion_matrix(y_test, predictions, labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix — Random Forest (AI Learn)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved outputs/confusion_matrix.png')

## Step 10 — Feature Importance Table & Graph

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)
importance_df['Rank'] = importance_df.index + 1
importance_df = importance_df[['Rank', 'Feature', 'Importance']]
importance_df['Importance'] = importance_df['Importance'].round(4)

print(importance_df.to_string(index=False))
importance_df.to_csv('outputs/feature_importance.csv', index=False)
print('\n✅ Saved outputs/feature_importance.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importance_df)))
bars = ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
               color=colors, edgecolor='grey', linewidth=0.5)
for bar, val in zip(bars, importance_df['Importance'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=9)
ax.set_title('Feature Importance — Random Forest (AI Learn Platform Data)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score'); ax.set_ylabel('Feature')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved outputs/feature_importance.png')

## Step 11 — Save Model + Label Encoder
Both files are needed in your FastAPI backend.

In [ ]:
joblib.dump(model, 'outputs/random_forest_v2.pkl')
joblib.dump(difficulty_encoder, 'outputs/difficulty_label_encoder.pkl')
joblib.dump(FEATURE_COLUMNS, 'outputs/feature_columns.pkl')

print('✅ Saved outputs/random_forest_v2.pkl')
print('✅ Saved outputs/difficulty_label_encoder.pkl')
print('✅ Saved outputs/feature_columns.pkl  (ensures correct feature order at prediction time)')
print(f'\nModel file size: {os.path.getsize("outputs/random_forest_v2.pkl")/1024:.1f} KB')

## Step 12 — Quick Inference Test
Simulates how FastAPI's `ml_service.py` will call the model.

In [ ]:
# Example: a struggling student on a Medium-difficulty topic
sample = pd.DataFrame([{
    'topic_id': 5,
    'quiz_accuracy': 35.0,
    'avg_time_per_question': 70.0,
    'total_attempts': 4,
    'question_difficulty_enc': difficulty_encoder.transform(['Medium'])[0],
    'videos_watched': 1,
    'articles_read': 0,
    'chatbot_questions': 3,
    'study_duration': 20.0,
    'daily_streak': 2,
    'ewma_accuracy': 38.0,
    'prerequisite_mastery': 45.0,
    'previous_attempt_accuracy': 30.0,
}])[FEATURE_COLUMNS]

pred = model.predict(sample)[0]
proba = model.predict_proba(sample)[0]
print(f'Prediction: {pred}')
print(f'Probabilities: {dict(zip(model.classes_, proba.round(3)))}')

## Step 13 — Download All Artifacts

In [ ]:
from google.colab import files

artifacts = [
    'outputs/random_forest_v2.pkl',
    'outputs/difficulty_label_encoder.pkl',
    'outputs/feature_columns.pkl',
    'outputs/confusion_matrix.png',
    'outputs/feature_importance.png',
    'outputs/feature_importance.csv',
    'outputs/classification_report.txt',
    'outputs/label_distribution.png',
]
for path in artifacts:
    if os.path.exists(path):
        files.download(path)
        print(f'⬇️  {path}')
    else:
        print(f'❌  Missing: {path}')

---
## ✅ Summary — Files for Backend Integration
| File | Purpose |
|---|---|
| `random_forest_v2.pkl` | The trained model (replaces xAPI model in `ml_service.py`) |
| `difficulty_label_encoder.pkl` | Encodes `question_difficulty` string → int at inference time |
| `feature_columns.pkl` | Exact column order the model expects — use this to build the feature vector |
| `confusion_matrix.png`, `feature_importance.png/csv`, `classification_report.txt` | Documentation / PPT / viva |

### Backend integration snippet
```python
import joblib
model = joblib.load('models/random_forest_v2.pkl')
difficulty_encoder = joblib.load('models/difficulty_label_encoder.pkl')
feature_columns = joblib.load('models/feature_columns.pkl')

# Build feature vector in the SAME order as feature_columns, then:
prediction = model.predict(feature_df[feature_columns])
```